# Get the FASTA and FASTQ formats of the Sequences

1. FASTA will allow me to run a BLAST.
2. FASTQ will allow me to run HISAT2 (bowtie2 doesn't account for splicing)

In [1]:
library(data.table)

In [8]:
table <- fread("/scratch/Shares/clauset/Clauset_ABNexus//Counting/Probe_prep/data/PanCancer_IO_360_Gene_List.txt", 
              skip=1, header=TRUE)
table[1:2,]

# get the Name to use
table$Name <- paste0(">", table[["Official Symbol"]], ":", table[["Accession"]])

Official Symbol,Accession,Alias / Previous Symbol,Target Sequence,Official Full Name,Other targets or Isoform Information,V7,V8
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>
A2M,NM_000014.4,"FWP007,S863-7,CPAMD5",TCCATCTCAATCCCTGTGAAGTCAGACATTGCTCCTGTCGCTCGGTTGCTCATCTATGCTGTTTTACCTACCGGGGACGTGATTGGGGATTCTGCAAAAT,alpha-2-macroglobulin,,NA,NA
ACVR1C,NM_145259.2,"activin A receptor, type IC;ALK7,ACVRLK7",GGAATTTTGCCACCATGTGACTTATTGGGGCAGAGAAAACTCAGGGTTGTCTTTGAGTCTGCACAAAAGCACCAGGGAACCTGCTTAGCAAATCGTCTGA,activin A receptor type 1C,,NA,NA


In [14]:
dim(table)
table <- table[table[["Accession"]] != "",]

dim(table)

[1] 772   9

[1] 770   9

In [15]:
# Get the sequences
write_df <- table[,c("Name", "Target Sequence")]
lines_vector <- as.vector(t(write_df))
lines_vector[1:5]

[1] ">A2M:NM_000014.4"                                                                                    
[2] "TCCATCTCAATCCCTGTGAAGTCAGACATTGCTCCTGTCGCTCGGTTGCTCATCTATGCTGTTTTACCTACCGGGGACGTGATTGGGGATTCTGCAAAAT"
[3] ">ACVR1C:NM_145259.2"                                                                                 
[4] "GGAATTTTGCCACCATGTGACTTATTGGGGCAGAGAAAACTCAGGGTTGTCTTTGAGTCTGCACAAAAGCACCAGGGAACCTGCTTAGCAAATCGTCTGA"
[5] ">ADAM12:NM_003474.5"

In [16]:
fasta_file = "/scratch/Shares/clauset/Clauset_ABNexus//Counting/Probe_prep/data/PanCancer_IO_360_Probe_Sequences.fa"
writeLines(lines_vector, fasta_file)

### Get the FASTQ from the fasta

In [13]:
# Load necessary library
if (!require("Biostrings")) install.packages("Biostrings")
library(Biostrings)

Loading required package: Biostrings

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, table,
    tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:data.table’:

    first, second


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loadin

In [21]:


# Read the FASTA file
fasta_sequences <- readDNAStringSet(fasta_file)

# Define dummy quality scores (e.g., all 'I' which corresponds to 40 in Phred)
quality_scores <- sapply(fasta_sequences, function(seq) paste(rep("I", nchar(seq)), collapse = ""))
quality_scores[1:5]
table$quality_score <- quality_scores
table$fastq_name <- paste0("@", table[["Official Symbol"]], ":", table[["Accession"]])
table$third_line <- "+"
write_df = table[,c("fastq_name", "Target Sequence", "third_line", "quality_score")]
lines_vector <- as.vector(t(write_df))
lines_vector[1:4]
# Create a FASTQ file
fastq_file <- "/scratch/Shares/clauset/Clauset_ABNexus//Counting/Probe_prep/data/PanCancer_IO_360_Probe_Sequences.fastq"
writeLines(lines_vector, fastq_file)

A2M:NM_000014.4 
"IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII" 
                                                                                    ACVR1C:NM_145259.2 
"IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII" 
                                                                                    ADAM12:NM_003474.5 
"IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII" 
                                                                                 ADGRE1:NM_001256252.1 
"IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII" 
                                                                                       ADM:NM_001124.2 
"IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII"

[1] "@A2M:NM_000014.4"                                                                                    
[2] "TCCATCTCAATCCCTGTGAAGTCAGACATTGCTCCTGTCGCTCGGTTGCTCATCTATGCTGTTTTACCTACCGGGGACGTGATTGGGGATTCTGCAAAAT"
[3] "+"                                                                                                   
[4] "IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII"